In [10]:
categorical_cols = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]
numerical_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
target_colm = "Churn"

In [11]:
import pandas as pd
from pandas import DataFrame

In [12]:
filepath = "../data/Telco-Customer-Churn.csv"
raw_dataset = pd.read_csv(filepath)

In [ ]:
from sklearn.model_selection import train_test_split

raw_dataset.drop(columns='customerID')
raw_dataset["TotalCharges"] = pd.to_numeric(raw_dataset['TotalCharges'].replace(" ", 0))
y = raw_dataset['Churn']
X = raw_dataset.drop(columns='Churn')

# Split your features (X) and target (y) into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [14]:
print(type(X_train))

<class 'pandas.core.frame.DataFrame'>


In [15]:
import torch.nn as nn
import torch

class ChurnClassifier01(nn.Module):
    def __init__(self, input_dim=46, hidden_dim=16, hidden_dim_02=8):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),  # <-- 4 not 20
            nn.ReLU(),

            nn.Linear(hidden_dim, hidden_dim_02),
            nn.ReLU(),

            nn.Linear(hidden_dim_02, 1),
            nn.Sigmoid()
        )

    def forward(self, X):
        return self.net(X.float())

In [22]:
import torch
import torch.nn as nn


class ChurnClassifier(nn.Module):
    def __init__(
        self,
        input_dim: int = 46,
        hidden_dim_01: int = 256,
        hidden_dim_02: int = 512,
        hidden_dim_03: int = 256,
        hidden_dim_04: int = 64,
        output_dim: int = 1,
    ) -> None:
        super().__init__()

        self.net = nn.Sequential(
            # Input layer to first hidden layer
            nn.Linear(input_dim, hidden_dim_01),
            nn.ReLU(),
            # 1st hidden layer to 2nd hidden layer
            nn.Linear(hidden_dim_01, hidden_dim_02),
            nn.ReLU(),
            # 2nd hidden layer to 3rd hidden layer
            nn.Linear(hidden_dim_02, hidden_dim_03),
            nn.ReLU(),
            # 3rd hidden layer to 4th hidden layer
            nn.Linear(hidden_dim_03, hidden_dim_04),
            nn.ReLU(),
            # 4th hidden layer to 5th hidden layer
            nn.Linear(hidden_dim_04, output_dim),
            # nn.Sigmoid(),
        )

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return self.net(X.float())


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder
from skorch import NeuralNetBinaryClassifier


# Feature preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

# Encode target separately
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train).astype("float32")



# Wrap the PyTorch model using Skorch
# Note: Skorch handles the training loop, epochs, and optimizer under the hood!
pytorch_skorch_wrapper = NeuralNetBinaryClassifier(
    module=ChurnClassifier,
    module__input_dim=46,          # Passes this argument to ChurnClassifier's __init__
    criterion=nn.BCEWithLogitsLoss, # Standard loss for binary classification
    optimizer=torch.optim.Adam,
    lr=0.001,
    max_epochs=100,
    batch_size=64,
    train_split=None               # Disables skorch's internal validation split if you want to use sklearn's
)

# 4. Bind them together into one flawless Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('nn_model', pytorch_skorch_wrapper)
])

In [26]:
pipeline.fit(X_train, y_train_encoded)

  epoch    train_loss     dur
-------  ------------  ------
      1        0.4699  0.9377
      2        0.4224  0.8001
      3        0.4152  0.7880
      4        0.4100  0.5741
      5        0.4055  0.5752
      6        0.3997  0.5862
      7        0.3953  0.5567
      8        0.3904  0.7207
      9        0.3847  0.6944
     10        0.3782  0.6854
     11        0.3753  0.6903
     12        0.3686  0.6618
     13        0.3584  0.6787
     14        0.3507  0.6541
     15        0.3452  0.6519
     16        0.3407  0.7319
     17        0.3276  0.7168
     18        0.3193  0.7613
     19        0.3146  0.7467
     20        0.2974  0.7493
     21        0.2989  0.7142
     22        0.2869  0.6709
     23        0.2765  0.6906
     24        0.2641  0.7010
     25        0.2410  0.8937
     26        0.2345  0.9863
     27        0.2246  1.1540
     28        0.2161  0.9760
     29        0.2099  0.9338
     30        0.1994  0.7579
     31        0.1995  0.6755
     32   

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('nn_model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['customerID','gender','SeniorCitizen',...,'PaymentMethod', 'MonthlyCharges','TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remaind